In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# TOOL 1
@tool
def calculate(expression: str) -> str:
    """Calcola un'espressione matematica semplice."""
    try:
        return str(eval(expression))
    except Exception:
        return "Errore nel calcolo."


# TOOL 2
@tool
def fictional_weather(city: str) -> str:
    """Restituisce un bollettino meteo completamente fittizio per una città."""
    
    weather = {
        "Milano": "18°C, pioggia leggera, vento 12 km/h",
        "Roma": "27°C, soleggiato, vento 8 km/h",
        "Bergamo": "16°C, nuvoloso, vento 10 km/h",
    }

    return weather.get(
        city,
        f"{city}: 22°C, parzialmente nuvoloso, vento 7 km/h"
    )


# MODELLO
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    max_tokens=1000
)


# BIND DEI TOOL
llm_with_tools = llm.bind_tools([
    calculate,
    fictional_weather
])

response = llm_with_tools.invoke("Qual è il tempo a Desenzano e quanto fa 5 + 3?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)


conversation = [HumanMessage("Qual è il tempo a Desenzano e quanto fa 5 + 3?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "calculate":
        result = calculate.invoke(call["args"])
    elif call["name"] == "fictional_weather":
        result = fictional_weather.invoke(call["args"])
    conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print("content:", repr(final.content))



content: ''
tool_calls: [{'name': 'fictional_weather', 'args': {'city': 'Desenzano'}, 'id': 'chatcmpl-tool-a73213760ce65805', 'type': 'tool_call'}]
content: '**Tempo a Desenzano:** 22\u202f°C, parzialmente nuvoloso, vento 7\u202fkm/h.  \n\n**Somma richiesta:** \\(5 + 3 = 8\\).'
